Import Libraries

In [137]:
library(ggplot2)
library(caTools)
library(dplyr)

Create Synthetic Dataset

In [138]:
set.seed(123)
n <- 1000

In [139]:
fico_score <- runif(n, 450, 850)
orig_ltv <- runif(n, 50, 120)
dti <- runif(n, 10, 65)
income <- runif(n, 30000, 175000)


In [140]:
#Create premium with realistic relationships and some noise
risk_score <- 
        (850 - fico_score) * 0.001 +
        (orig_ltv-50) * 0.0035 +
        (dti-10) * 0.007 +
        (175000-income)/1000 * 0.0001 +
        rnorm(n, 0, 0.1)  # Add this random noise


In [141]:
early_default <- ifelse(risk_score > median(risk_score)*1.25, 1, 0)

In [142]:
loan_data <- data.frame(fico_score = round(fico_score, 0), orig_ltv = round(orig_ltv, 0), dti = round(dti, 0), income =round(income, 0), early_default = early_default)

In [143]:
#Check the distribution
table(early_default)
cat("Default rate:", round(mean(early_default) * 100, 1), "%")

early_default
  0   1 
736 264 

Default rate: 26.4 %

Running Logistic Regression

In [144]:
logisticReg <- loan_data[c("fico_score", "early_default")]

In [145]:
glm_r <- glm(formula = early_default ~ fico_score, family = binomial, data=logisticReg)

In [146]:
# Store for easy reference
intercept <- glm_r$coefficients[1]
fico_coef <- glm_r$coefficients[2]

cat("Intercept:", round(intercept, 2), "\n")
cat("Fico Score Coefficient:", round(fico_coef, 4), "\n")

Intercept: 6.49 
Fico Score Coefficient: -0.0122 


In [147]:
odds_ratio <- exp(fico_coef)
cat("Odds Ratio:", round(odds_ratio, 4), "\n")

Odds Ratio: 0.9879 


In [148]:
# Calculate probabilities for specific FICO scores  
fico_550 <- data.frame(fico_score = 550)
fico_650 <- data.frame(fico_score = 650)
fico_750 <- data.frame(fico_score = 750)

prob_550 <- predict(glm_r, newdata = fico_550, type = "response")
prob_650 <- predict(glm_r, newdata = fico_650, type = "response") 
prob_750 <- predict(glm_r, newdata = fico_750, type = "response")

cat("550 FICO default probability:", round(prob_550 * 100, 1), "%\n")
cat("650 FICO default probability:", round(prob_650 * 100, 1), "%\n")
cat("750 FICO default probability:", round(prob_750 * 100, 1), "%\n")

550 FICO default probability: 44.1 %
650 FICO default probability: 18.8 %
750 FICO default probability: 6.4 %


In [149]:
# Create a small data frame for the specific predictions
prediction_points <- data.frame(
  fico_score = c(550, 650, 750),
  probability = c(prob_550, prob_650, prob_750),
  min_val = c(0, 0, 0),
  max_val = c(prob_550*1.5, prob_650*1.75, prob_750*2),
  labels = c(
              paste("High Risk: ",round(prob_550*100, 1),"%"),
              paste("Med Risk: ",round(prob_650*100, 1),"%"),
              paste("Low Risk: ",round(prob_750*100, 1),"%")
  )
  )

In [150]:
# Simple approach - plot your existing data with regression curve
log_plot <- 
ggplot(trainingset, aes(x = fico_score, y = early_default)) +
  geom_point(alpha = 0.3) +
  geom_smooth(method = "glm", method.args = list(family = "binomial"), se = FALSE) +
  geom_point(data = prediction_points, aes(x = fico_score, y = probability), 
             color = "red", size = 3) +
  # Add vertical lines from min to max values
  geom_segment(data = prediction_points, 
               aes(x = fico_score, xend = fico_score, 
                   y = min_val, yend = max_val), 
               color = "red", linewidth = 1) +
  geom_text(data=prediction_points,
            aes(x=fico_score+35, y=probability*1.25, label=labels),
            color="darkred", , size = 4.5, hjust = 0.5)
  labs(title = "Early Default Probability by FICO Score",
       x = "FICO Score",
       y = "Early Default Probability") +
  theme_minimal()

  # Save a plot with specific dimensions
ggsave("log_plot.png", log_plot, width = 12, height = 8, units = "in", dpi = 300)


NULL

`geom_smooth()` using formula = 'y ~ x'


Multiple Logistic Regression

In [151]:
multLogisticReg <- loan_data

In [152]:
m_glm_r <- glm(formula = early_default ~ fico_score + orig_ltv + dti + income, family = binomial, data=multLogisticReg)

In [153]:
# Extract coefficients for interpretation
intercept <- m_glm_r$coefficients[1]
fico_coef <- m_glm_r$coefficients[2]
ltv_coef <- m_glm_r$coefficients[3]
dti_coef <- m_glm_r$coefficients[4]
income_coef <- m_glm_r$coefficients[5]

cat("FICO Score Coefficient:", round(fico_coef, 4), "\n")
cat("LTV Coefficient:", round(ltv_coef, 4), "\n")
cat("DTI Coefficient:", round(dti_coef, 4), "\n")
cat("Income Coefficient:", round(income_coef, 6), "\n")

FICO Score Coefficient: -0.0202 


LTV Coefficient: 0.0569 
DTI Coefficient: 0.1138 
Income Coefficient: 1e-06 


In [160]:
# Compare predictions for sample borrowers
sample_borrower1 <- data.frame(fico_score = 700, orig_ltv = 70, dti = 35, income = 90000)
sample_borrower2 <- data.frame(fico_score = 700, orig_ltv = 100, dti = 35, income = 90000)

# Single vs multiple model predictions
prob_single_1 <- predict(glm_r, newdata = sample_borrower1, type = "response")
prob_multiple_1 <- predict(m_glm_r, newdata = sample_borrower1, type = "response")

prob_single_2 <- predict(glm_r, newdata = sample_borrower2, type = "response")
prob_multiple_2 <- predict(m_glm_r, newdata = sample_borrower2, type = "response")

cat("Good borrower (700 FICO, 70% LTV, 35% DTI):\n")
cat("Single variable model:", round(prob_single_1 * 100, 1), "%\n")
cat("Multiple variable model:", round(prob_multiple_1 * 100, 1), "%\n")

cat("Good borrower (700 FICO, 100% LTV, 35% DTI):\n")
cat("Single variable model:", round(prob_single_2 * 100, 1), "%\n")
cat("Multiple variable model:", round(prob_multiple_2 * 100, 1), "%\n")

Good borrower (700 FICO, 70% LTV, 35% DTI):
Single variable model: 11.2 %
Multiple variable model: 1 %
Good borrower (700 FICO, 100% LTV, 35% DTI):
Single variable model: 11.2 %
Multiple variable model: 5.5 %
